In [1]:
import pandas as pd
import re

In [2]:
df=pd.read_csv("IMDB-Dataset.csv")
print(df.head())
print(df.shape)
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [3]:
df['label']=df['sentiment'].map({'positive':1,'negative':0})    #Convert label into 0/1
def clean_text(text):
    text=text.lower()
    text=re.sub(r'<br\s*/?>','',text)
    text = re.sub(r'[^a-z\s]', ' ', text)         
    text = re.sub(r'\s+', ' ', text).strip()      
    return text
    
df['clean_review'] = df['review'].apply(clean_text)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
X_train, X_test,y_train,y_test=train_test_split(
    df['clean_review'],df['label'],test_size=0.2, random_state=42,stratify=df['label']
)
print(X_train.shape, X_test.shape)

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train) 
X_test_tfidf = vectorizer.transform(X_test)          

print(X_train_tfidf.shape, X_test_tfidf.shape)


(40000,) (10000,)
(40000, 5000) (10000, 5000)


In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
nb_preds = nb.predict(X_test_tfidf)

print("Naive Bayes accuracy:", accuracy_score(y_test, nb_preds))
print(classification_report(y_test, nb_preds))

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
lr_preds = lr.predict(X_test_tfidf)

print("Logistic Regression accuracy:", accuracy_score(y_test, lr_preds))
print(classification_report(y_test, lr_preds))

Naive Bayes accuracy: 0.8545
              precision    recall  f1-score   support

           0       0.86      0.85      0.85      5000
           1       0.85      0.86      0.86      5000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

Logistic Regression accuracy: 0.8927
              precision    recall  f1-score   support

           0       0.90      0.89      0.89      5000
           1       0.89      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [6]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000))
])

# Note: fit on raw text (X_train), not X_train_tfidf!
pipeline.fit(X_train, y_train)

preds = pipeline.predict(X_test)
print("Pipeline accuracy:", accuracy_score(y_test, preds))

Pipeline accuracy: 0.8927


In [7]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'tfidf__max_features': [5000, 10000],
    'tfidf__ngram_range': [(1,1), (1,2)],   # unigrams only, vs unigrams+bigrams
    'clf__C': [0.1, 1, 10]                   # regularization strength
}

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

best_model = grid.best_estimator_
test_preds = best_model.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, test_preds))

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best params: {'clf__C': 10, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}
Best CV score: 0.8926999742415189
Test accuracy: 0.9001


In [8]:
import numpy as np
from sklearn.metrics import confusion_matrix

# Confusion matrix
cm = confusion_matrix(y_test, test_preds)
print(cm)

# Most influential words
tfidf = best_model.named_steps['tfidf']
clf = best_model.named_steps['clf']

feature_names = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_[0]

top_positive_idx = np.argsort(coefs)[-15:][::-1]
top_negative_idx = np.argsort(coefs)[:15]

print("Top POSITIVE words:")
for i in top_positive_idx:
    print(f"  {feature_names[i]:<20} {coefs[i]:.3f}")

print("\nTop NEGATIVE words:")
for i in top_negative_idx:
    print(f"  {feature_names[i]:<20} {coefs[i]:.3f}")

[[4488  512]
 [ 487 4513]]
Top POSITIVE words:
  great                11.197
  excellent            10.575
  hilarious            10.357
  perfect              10.343
  amazing              9.365
  well worth           8.713
  wonderful            8.666
  enjoyable            8.402
  refreshing           8.219
  brilliant            8.092
  loved this           7.550
  superb               7.542
  incredible           7.459
  wonderfully          7.322
  today                7.240

Top NEGATIVE words:
  awful                -14.940
  worst                -14.708
  waste                -13.153
  boring               -12.385
  poorly               -11.621
  disappointment       -11.462
  poor                 -10.866
  disappointing        -10.550
  horrible             -10.387
  lacks                -10.254
  dull                 -9.877
  bad                  -9.832
  terrible             -9.810
  not worth            -9.626
  the worst            -9.396
